# Channel Selection by Random Forest Feature Importance

**Dataset**: MOABB BNCI2014-001 (Motor Imagery)  
**Channels**: 22 channels  
**Sampling rate**: 250 Hz  
**Subject**: 1

---

## Overview

We train a Random Forest and extract feature importances to select channels.

## Expected outputs

- Bar chart of feature importances sorted descending
- Topomap with dot size representing importance

## Key parameters

| Parameter | Value |
| --- | --- |
| n_estimators | 100 |
| N_SELECT | 10 |


## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB).


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


## 3. Explore the data


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


## 4. Train Random Forest and extract importance


In [ ]:
from scipy.signal import welch
from sklearn.ensemble import RandomForestClassifier

FS = 250
N_SELECT = 10
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(features, labels)
importances = clf.feature_importances_
print(f'Top 5 features: {np.argsort(importances)[::-1][:5]}')


## 5. Interactive plot

**What to look for:**

- Importance is spread across channels with concentration in central region
- Dot size represents importance level


In [ ]:
import plotly.graph_objects as go

sorted_feat = np.argsort(importances)[::-1]
colors = ['green' if i < N_SELECT else 'gray' for i in range(len(importances))]
fig = go.Figure(go.Bar(x=[str(i) for i in sorted_feat], y=importances[sorted_feat], marker_color=colors, name='Importance'))
fig.update_layout(height=500, title='Random Forest Feature Importance', xaxis_title='Feature (sorted)', yaxis_title='Importance')
fig.show()


## What did we learn?

- Random Forest gives importance to each feature in a single training
- Faster than RFE but less accurate as it does not consider interactions iteratively
- Importance reflects the feature's contribution to reducing uncertainty
